## What a registry actually is

A **container registry** is a web service that stores and serves images over a standard HTTP API — the **OCI Distribution Specification**. Every registry — Docker Hub, GHCR, ECR, your own — speaks the same protocol, which is why the `docker` CLI doesn't care *which* one it's talking to; it just speaks the standard.

A registry stores exactly two kinds of thing:

1. **Blobs** — content-addressable byte storage. Every layer is a blob; every image config is a blob; each is identified by its **SHA-256 digest** (`sha256:abc…`). Upload the same bytes twice and the second upload is a **no-op** — the registry already has that digest.
2. **Manifests** — small JSON documents describing one image: the list of layer digests, the config digest, the platform. The manifest itself has a digest, and **a "tag" is just a mutable pointer from a name like `1.27.0` to a manifest digest.**

So a `docker push` is really four steps:

```
1. Upload each layer blob   (deduped by digest — only new layers cross the wire)
2. Upload the config blob
3. Upload the manifest that lists them
4. Point the tag at the manifest's digest
```

A pull is the same steps reversed. There's no magic compression or diffing — just content-addressed blobs and a JSON manifest tying them together.

Two consequences fall out of this design, and they explain most of the module:

- **Dedup is free.** Push a new build that shares 90% of its layers with the last one and only the changed 10% uploads — the rest already exist by digest. That's why `push`/`pull` are often far faster than the image size suggests.
- **Digest = identity.** The tag is a movable label; the manifest digest is the immutable truth. Pin by digest and you get *exactly* those bytes, forever — the foundation for reproducible deploys later in the module.